# 发布反馈

### 设置

In [ ]:
# 你可以在代码中直接设置
import os
os.environ["LANGSMITH_API_KEY"] = ""
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "langsmith-academy"

In [ ]:
# 或者你可以使用 .env 文件
from dotenv import load_dotenv
load_dotenv(dotenv_path="../../.env", override=True)

### 为运行添加反馈

只要你知道 run_id，就可以通过编程方式为现有的运行添加反馈。让我们从 LangSmith UI 中获取一个 run_id 并在这里添加。

In [ ]:
run_id = ""

现在，让我们添加一些连续性反馈

In [ ]:
from langsmith import Client

client = Client()

client.create_feedback(
  run_id,
  key="sample-continuous",
  score=7.0,
  comment="这是一个示例连续性反馈",
)

现在，让我们也添加一些分类反馈！

In [ ]:
from langsmith import Client

client = Client()

client.create_feedback(
  run_id,
  key="sample-categorical",
  value="no",
  comment="这是一个示例分类反馈",
)

### 预生成运行 ID 以获取反馈

![Generate_run_id](../../images/generate_run_id.png)

使用 LangChain，我们提供了在代码被调用和生成运行 ID 之前预生成和定义运行 ID 的能力。通过这个功能，你可以在初始生成之前访问你的运行 ID，这对于发送反馈等操作非常有用。下面的示例演示了这一点。

In [ ]:
import uuid

pre_defined_run_id = uuid.uuid4()
pre_defined_run_id

In [ ]:
from langsmith import traceable

@traceable
def foo():
    return "这是一个示例运行！"

我们通过 `langsmith_extra` 在函数调用中传递一个配置，其中包含我们预定义的 run_id

In [ ]:
foo(langsmith_extra={"run_id": pre_defined_run_id})

现在我们可以直接为这个运行创建反馈！

In [ ]:
from langsmith import Client

client = Client()

client.create_feedback(pre_defined_run_id, "user_feedback", score=1)

### 预签名反馈 URL

![presigned url](../../images/presigned_url.png)


这对于预签名反馈 URL 也很有用。当你无法向客户端暴露 API 密钥或其他机密时（例如在 Web 应用程序中），你会想要使用这些。使用预定义的 run_id，LangSmith 有一个端点 create_presigned_feedback_token，它将创建一个用于发送反馈的 URL，而不需要使用机密。

In [ ]:
pre_signed_url_id = uuid.uuid4()
pre_signed_url_id

In [ ]:
pre_signed_url = client.create_presigned_feedback_token(pre_signed_url_id, "user_presigned_feedback")

print(pre_signed_url)

在这里，我们可以看到即使我们还没有创建运行，我们仍然能够生成反馈 URL。

现在，让我们调用我们的链，以便创建具有该 ID 的运行：

In [ ]:
foo(langsmith_extra={"run_id": pre_signed_url_id})

然后，一旦我们的运行被创建，我们就可以使用反馈 URL 发送反馈：

In [ ]:
import requests

url_with_score = f"{pre_signed_url.url}?score=1"

response = requests.get(url_with_score)

if response.status_code >= 200 and response.status_code < 300:
    print("反馈提交成功！")
else:
    print("反馈提交失败！")